In [1]:
from utils import YOLO11KDTrainer
from utils import YOLO11KDLoss
from ultralytics import YOLO

In [ ]:
def main():

    teacher = YOLO(r"C:\Users\kccistc1\Desktop\yolo\train\runs\detect\train_baseline_reduced\yolo11m_train_baseline_reduced\weights\best.pt")
    teacher_model = teacher.model
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad = False

    reg_max = teacher_model.model[-1].reg_max
    print(f"[KD] reg_max={reg_max}, box_ch={4 * reg_max}")

    kd_loss_fn = YOLO11KDLoss(
        reg_max     = reg_max,
        temperature = 8.0,
        alpha       = 0.4,
        beta        = 0.3,
        gamma       = 0.1
    )
    overrides = dict(
        data     = "../Train_Baseline_Reduced/data.yaml",
        epochs   = 150,
        imgsz    = 416,
        batch    = 16,
        project  = "train_kd_from_11m",
        name     = "baseline_reduced_8_0-4_0-3_0-1",
        device   = "0",
        patience = 25,
        model    = "yolo11n.pt",
    )

    trainer = YOLO11KDTrainer(
        teacher_model = teacher_model,
        kd_loss_fn    = kd_loss_fn,
        overrides     = overrides
    )
    trainer.train()

if __name__ == "__main__":
    main()

[KD] reg_max=16, box_ch=64
Ultralytics 8.4.37  Python-3.10.20 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Train_Baseline_Reduced/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_kd_from_11s24, nbs=64, nms=False, opset=None, optim

In [3]:
model = YOLO(r"runs\detect\train_kd\yolo11n_kd_from_11s24\weights\best.pt")

metrics = model.val(
    data="../test_data/data.yaml",
    split="val"
)

Ultralytics 8.4.37  Python-3.10.20 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 90.837.7 MB/s, size: 36.6 KB)
val: Scanning C:\Users\kccistc1\Desktop\yolo\train\test_data\valid\labels.cache... 306 images, 179 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 306/306  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 15.2it/s 1.3s0.1s
                   all        306        614      0.901      0.827      0.847      0.472
               soldier         79        274      0.861      0.745      0.771      0.358
                  tank        108        340      0.941      0.909      0.924      0.585
Speed: 0.6ms preprocess, 1.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\kccistc1\Desktop\yolo\train\knowledge_distillation\runs\detect\val5


In [4]:
metrics = model.val(
    data="../test_data_reduced/data.yaml",
    split="val"
)


Ultralytics 8.4.37  Python-3.10.20 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
val: Fast image access  (ping: 0.20.1 ms, read: 52.315.2 MB/s, size: 26.3 KB)
val: Scanning C:\Users\kccistc1\Desktop\yolo\train\test_data_reduced\valid\labels.cache... 277 images, 178 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 277/277  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 10.0it/s 1.8s0.2s
                   all        277        454      0.915      0.847      0.875      0.508
               soldier         53        195      0.872      0.771      0.813      0.403
                  tank         82        259      0.957      0.923      0.938      0.612
Speed: 0.9ms preprocess, 2.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to C:\Users\kccistc1\Desktop\yolo\train\knowledge_distillation\runs\detect\val6
